# Análisis exploratorio de datos (EDA): rendimiento en matemáticas

**Dataset:** *Student Performance*, archivo `student-mat.csv` (UCI Machine Learning Repository)

**Autor:** Felipe Giraldo Zapata

## Preparación

In [ ]:
import matplotlib.pyplot as plt   # base de los gráficos: títulos, ejes, leyendas
import pandas as pd               # tablas de datos (DataFrame)
import plotly.express as px       # gráficos interactivos
import seaborn as sns             # gráficos estadísticos sobre matplotlib

# Estilo común para todos los gráficos de seaborn: fondo blanco con cuadrícula
sns.set_theme(style="whitegrid")

## Paso 1. Entender el dataset

* **Nombre y fuente:** *Student Performance*, de P. Cortez y A. Silva (2008), publicado en el
  [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/320/student+performance). Los datos
  vienen de reportes escolares y encuestas a estudiantes de dos colegios públicos de Portugal.
* **Qué es una fila:** un estudiante de secundaria en la asignatura de **matemáticas**. Existe otro archivo
  (`student-por.csv`) para la asignatura de portugués, que no se usa aquí.
* **Tamaño:** 395 filas y 33 columnas.
* **Grupos de columnas:**
  * *Quién es el estudiante:* `school` (GP = Gabriel Pereira, MS = Mousinho da Silveira), `sex`, `age`,
    `address` (U = urbana, R = rural).
  * *Su familia:* `famsize`, `Pstatus` (padres juntos o separados), `Medu` y `Fedu` (educación de la madre y del
    padre, de 0 = ninguna a 4 = superior), `Mjob`, `Fjob`, `guardian`, `famrel`.
  * *Su vida escolar:* `reason`, `traveltime`, `studytime`, `failures` (materias perdidas antes), `schoolsup`,
    `famsup`, `paid`, `activities`, `nursery`, `higher` (quiere estudios superiores), `absences`.
  * *Su vida social y salud:* `internet`, `romantic`, `freetime`, `goout`, `Dalc` y `Walc` (consumo de alcohol
    entre semana y el fin de semana, de 1 a 5), `health`.
  * *Sus notas:* `G1` y `G2` (primer y segundo periodo) y `G3` (nota final), en una escala de 0 a 20.
* **Variable objetivo:** `G3`, la nota final de matemáticas (0 a 20). En Portugal se aprueba con 10 o más, que
  es el umbral que usan Cortez y Silva. `G1` y `G2` son notas del mismo año, así que dicen mucho sobre `G3`.

## Paso 2. Cargar los datos

In [ ]:
# Datos originales: esta variable no se modifica en ningún paso
raw = pd.read_csv("data/raw/student-mat.csv")

# Copia de trabajo: todos los cambios se hacen sobre df
df = raw.copy()

El archivo se carga con las opciones por defecto, porque sus columnas están separadas por comas. La versión que
se descarga directamente de UCI usa punto y coma; con ese archivo habría que escribir
`pd.read_csv("data/raw/student-mat.csv", sep=";")`. `raw` queda intacto y todos los cambios se hacen sobre `df`.

## Paso 3. Primera mirada

In [ ]:
df.shape  # (número de filas, número de columnas)

In [ ]:
df.head()  # primeras 5 filas

In [ ]:
df.tail()  # últimas 5 filas

In [ ]:
df.info()  # tipo de dato y valores no vacíos por columna


* La tabla tiene **395 filas y 33 columnas**, como dice la página del dataset. El final del archivo son datos
  normales, sin totales ni notas.
* **17 columnas son texto** (`str`) y **16 son números enteros** (`int64`). Las 33 columnas tienen 395 valores no
  vacíos.
* **Números que en realidad son categorías:** 11 de las 16 columnas numéricas son escalas ordenadas guardadas
  como números: `Medu`, `Fedu`, `traveltime`, `studytime`, `failures`, `famrel`, `freetime`, `goout`, `Dalc`,
  `Walc` y `health`. Por ejemplo, en `studytime` el 2 significa "de 2 a 5 horas", no "2 horas". Su promedio sirve
  como resumen aproximado, pero no es una cantidad medida.
* **Columnas sí/no guardadas como texto:** 8 columnas (`schoolsup`, `famsup`, `paid`, `activities`, `nursery`,
  `higher`, `internet`, `romantic`) solo tienen `yes` o `no`.
* **Significado poco claro:** la documentación dice que `failures` vale *n* si 1 ≤ *n* < 3 y 4 en otro caso,
  pero en los datos el máximo es 3. No queda claro si el 3 significa "3" o "3 o más".
* **No hay columna de identificación:** no existe un código de estudiante, así que no se puede revisar si un
  mismo estudiante aparece dos veces con datos distintos.

## Paso 4. Calidad de los datos

In [ ]:
# .isna() marca cada celda vacía; la primera .sum() cuenta por columna y la segunda suma todas las columnas
df.isna().sum().sum()

In [ ]:
df.duplicated().sum()  # filas idénticas a una fila anterior

In [ ]:
# Para cada columna numérica, cuántas filas valen exactamente 0
ceros = (df.select_dtypes("number") == 0).sum()

# Muestra solo las columnas que tienen al menos un 0
ceros[ceros > 0]

In [ ]:
# Filtra los estudiantes con nota final 0
sin_nota = df[df["G3"] == 0]

# Resume sus notas, ausencias y materias perdidas
sin_nota[["G1", "G2", "G3", "absences", "failures"]].describe().round(2)

In [ ]:
# Una comparación (== 0) da True o False por fila; .sum() cuenta los True
print("estudiantes con G3 = 0:      ", len(sin_nota))
print("de ellos, con 0 ausencias:   ", (sin_nota["absences"] == 0).sum())
print("de ellos, con G2 = 0:        ", (sin_nota["G2"] == 0).sum())
print("ausencias = 0 en todo el df: ", (df["absences"] == 0).sum())



* **Faltantes y duplicados:** 0 valores faltantes y 0 filas duplicadas. No hay columna de identificación, así que
  no se pueden buscar estudiantes repetidos con datos distintos.
* **Faltantes escondidos en las notas:** **38 estudiantes (9.6 %) tienen `G3 = 0`**. Los 38 tienen **0
  ausencias**, y 13 de ellos también tienen `G2 = 0`. Su `G1` va de 4 a 12, así que sí tenían notas durante el
  año. Lo más probable es que no presentaran la evaluación final o que se retiraran, y que el 0 signifique "sin
  nota", no "sacó 0". Si es así, sus 0 ausencias tampoco serían un dato real.
* **Ausencias:** 115 estudiantes tienen 0 ausencias, y 38 de ellos son los estudiantes sin nota final. En esas 38
  filas el 0 puede significar "no se registró".
* **Tipos:** ningún tipo impide el análisis. Las 8 columnas sí/no son texto y las 11 escalas son números (paso 3);
  se documentan, pero no se cambian.
* **Qué se corrige:** nada. Los 38 ceros se mantienen en `df`, y en los pasos 8 y 10 se muestra cuánto cambian los
  resultados con y sin ellos.

## Paso 5. Estadística descriptiva

In [ ]:
df.describe().round(2).T  # media, desviación, mínimo, cuartiles y máximo de cada columna numérica

In [ ]:
df.describe(exclude="number").T  # categorías distintas, la más frecuente y su frecuencia

* **Columna con sesgo:** `absences` tiene media 5.71 y mediana 4, con un máximo de 75. Unos pocos estudiantes con
  muchas ausencias suben el promedio: sesgo a la derecha.
* **Sesgo al revés en la nota final:** `G3` tiene media 10.42 y mediana 11. Aquí la media queda **por debajo** de
  la mediana porque los 38 ceros del paso 4 la bajan.
* **Mínimos extraños:** `G2` y `G3` llegan a 0, mientras que `G1` no baja de 3. Es la huella de los estudiantes sin
  nota final.
* **Edad:** va de 15 a 22 años, con mediana 17; un estudiante de 22 años es poco común en secundaria (paso 7).
* **Categorías más frecuentes:** el colegio `GP` tiene 349 de los 395 estudiantes y `higher = yes` aparece 375
  veces: casi todos quieren estudios superiores. En `Fjob` la categoría más frecuente es `other` (217
  estudiantes).

## Paso 6. Una columna a la vez

In [ ]:
# binwidth=2: cada barra agrupa 2 valores de ausencias; kde=True agrega la curva de densidad
sns.histplot(data=df, x="absences", binwidth=2, kde=True)
plt.title("Estudiantes según número de ausencias")
plt.xlabel("Ausencias")
plt.ylabel("Estudiantes")
plt.show()

In [ ]:
round((df["absences"] == 0).mean(), 3)  # el promedio de True/False es la proporción de True

In [ ]:
# Caja = mitad central de las edades; línea interna = mediana; puntos = posibles atípicos
sns.boxplot(data=df, x="age")
plt.title("Edad de los estudiantes")
plt.xlabel("Edad (años)")
plt.show()

In [ ]:
df["Mjob"].nunique()  # número de categorías distintas (cardinalidad)

In [ ]:
df["Mjob"].value_counts(normalize=True).round(3)  # participación de cada categoría, de mayor a menor

In [ ]:
# y="Mjob" dibuja barras horizontales; order las ordena de la categoría más frecuente a la menos frecuente
sns.countplot(data=df, y="Mjob", order=df["Mjob"].value_counts().index)
plt.title("Estudiantes según el trabajo de la madre")
plt.xlabel("Estudiantes")
plt.ylabel("Trabajo de la madre")
plt.show()

In [ ]:
df["higher"].value_counts(normalize=True).round(3)  # participación de "yes" y "no"

In [ ]:
# Nombres de las columnas de texto
texto = df.select_dtypes(exclude="number").columns

# Una columna falla si pierde categorías al quitar espacios y pasar a minúsculas
fallan = [col for col in texto
          if df[col].nunique() != df[col].str.strip().str.lower().nunique()]
print("columnas de texto revisadas:", len(texto))
print("columnas con categorías repetidas:", fallan)

In [ ]:
sorted(df["Mjob"].unique())  # en orden alfabético, dos escrituras de una palabra quedan juntas

**Columnas numéricas**

* **`absences`:** la primera barra (0 y 1 ausencias) es la más alta, y **115 estudiantes (29.1 %) tienen
  exactamente 0 ausencias**. Después las barras bajan rápido y queda una cola larga hasta 75. Es la forma con sesgo a la derecha.
* **`age`:** la mediana es 17 años y la mitad de los estudiantes tiene entre 16 y 18. Hay un solo punto fuera de
  los bigotes, en 22 años.

**Columnas categóricas**

* **`Mjob`:** 5 categorías. La más frecuente es `other`, con 35.7 %, seguida de `services` (26.1 %); las dos
  juntas cubren 61.8 % de los estudiantes. `health` es la menos frecuente (8.6 %). Los estudiantes se reparten
  entre las cinco, sin una categoría que domine.
* **`higher`:** 2 categorías, y `yes` cubre 94.9 %. Solo 20 estudiantes no quieren estudios superiores: un grupo
  pequeño, que hay que mirar con cuidado cuando se compare con la nota.
* **Categorías escritas de más de una forma:** ninguna. En las 17 columnas de texto el número de categorías no
  cambia con `strip` y `lower`, y la lista ordenada de `Mjob` muestra cinco categorías bien escritas.

## Paso 7. Valores atípicos

In [ ]:
def limites_iqr(serie: pd.Series) -> dict:
    """Calcula los cuartiles, el IQR, los límites y el número de atípicos de una columna."""
    q1 = serie.quantile(0.25)          # primer cuartil
    q3 = serie.quantile(0.75)          # tercer cuartil
    iqr = q3 - q1                      # ancho de la mitad central
    inferior = q1 - 1.5 * iqr          # límite inferior
    superior = q3 + 1.5 * iqr          # límite superior
    # | significa "o": cuenta los valores por debajo del límite inferior o por encima del superior
    atipicos = ((serie < inferior) | (serie > superior)).sum()
    return {"Q1": q1, "Q3": q3, "IQR": iqr, "inferior": inferior,
            "superior": superior, "atípicos": atipicos}


# Columnas que son cantidades (no escalas de 1 a 5)
columnas = ["age", "absences", "failures", "G1", "G2", "G3"]

# Aplica la función a cada columna y arma una tabla con una fila por columna
pd.DataFrame({col: limites_iqr(df[col]) for col in columnas}).T

In [ ]:
# .loc[filas, columnas]: filas con más de 20 ausencias y solo las columnas útiles para juzgarlas
df.loc[df["absences"] > 20, ["age", "failures", "absences", "G1", "G2", "G3"]].sort_values("absences")

In [ ]:
df.loc[df["age"] > 21, ["age", "failures", "absences", "G1", "G2", "G3"]]  # edades sobre el límite de 21

* **`absences`:** 15 atípicos, entre 21 y 75 ausencias. Parecen **casos reales y raros**, no errores: la
  documentación permite hasta 93 ausencias y sus notas son normales (el estudiante con 75 ausencias tiene
  `G3 = 9`). Son estudiantes con ausentismo alto.
* **`age`:** 1 atípico, el estudiante de 22 años. Tiene 3 materias perdidas antes, así que probablemente es un
  caso real de repitencia.
* **`G2`:** 13 atípicos, y son exactamente los 13 ceros del paso 4. No son estudiantes con un rendimiento raro,
  sino filas sin nota.
* **`G3`:** **0 atípicos**, aunque hay 38 ceros. El límite inferior es −1, así que la regla no ve los ceros.
* **`failures`:** el IQR es 0, porque 79 % de los estudiantes tiene 0 materias perdidas. Cualquier valor mayor que
  0 sale como atípico (83 estudiantes). En columnas discretas con un valor dominante la regla no sirve.
* **`G1`:** 0 atípicos.

## Paso 8. La variable objetivo

In [ ]:
# discrete=True: una barra por cada nota entera de 0 a 20
sns.histplot(data=df, x="G3", discrete=True)

# Línea vertical entre 9 y 10, el umbral de aprobación
plt.axvline(9.5, color="black", linestyle="--", label="aprueba desde 10")
plt.title("Estudiantes según nota final (G3)")
plt.xlabel("Nota final (0 a 20)")
plt.ylabel("Estudiantes")
plt.legend()
plt.show()

In [ ]:
# Estudiantes con nota final mayor que 0 (se usa también en el paso 10)
con_nota = df[df["G3"] > 0]

# Tabla con la media, la mediana y el número de estudiantes de cada grupo
pd.DataFrame({
    "todos": [df["G3"].mean(), df["G3"].median(), len(df)],
    "sin G3 = 0": [con_nota["G3"].mean(), con_nota["G3"].median(), len(con_nota)],
}, index=["media", "mediana", "estudiantes"]).round(2)

In [ ]:
# G3 >= 10 da True o False; .map() los traduce a "sí" y "no"
df["aprobado"] = (df["G3"] >= 10).map({True: "sí", False: "no"})

# Participación de cada categoría de la nueva columna
df["aprobado"].value_counts(normalize=True).round(3)

* **Distribución:** la nota final tiene una forma de campana entre 4 y 20, con la barra más alta en 10 (56
  estudiantes), y una barra separada en 0 con **38 estudiantes**. No hay ninguna nota entre 1 y 3: los ceros no son
  la cola de la campana, son otro grupo.
* **Media y mediana:** con todos los estudiantes, la media es 10.42 y la mediana 11. Sin los ceros, la media sube
  a 11.52 y la mediana sigue en 11. Los ceros bajan el promedio en más de un punto.
* **Aprobación:** **67.1 % aprueba** (`G3` ≥ 10) y 32.9 % no. Las clases están algo desbalanceadas: 1 de cada 3
  estudiantes no aprueba. Ese 67.1 % es la referencia del paso 9: un grupo con una tasa muy por debajo aprueba
  menos que el estudiante promedio.

## Paso 9. Dos columnas a la vez

### Numérica y numérica: `G2` y `G3`

In [ ]:
# alpha=0.4 hace los puntos semitransparentes: donde se superponen, el color es más oscuro
sns.scatterplot(data=df, x="G2", y="G3", alpha=0.4)
plt.title("Nota final contra nota del segundo periodo")
plt.xlabel("G2 (segundo periodo)")
plt.ylabel("G3 (nota final)")
plt.show()

### Categórica y numérica: `failures` y `G3`

In [ ]:
# Una caja de G3 por cada valor de failures (0, 1, 2 y 3)
sns.boxplot(data=df, x="failures", y="G3")
plt.title("Nota final según materias perdidas antes")
plt.xlabel("Materias perdidas antes (failures)")
plt.ylabel("G3 (nota final)")
plt.show()

In [ ]:
df.groupby("failures")["G3"].agg(["median", "count"])  # mediana de G3 y número de estudiantes por grupo

### Categórica y categórica: `higher` y `aprobado`

In [ ]:
# Filas = higher, columnas = aprobado; normalize="index" convierte cada fila en proporciones que suman 1
higher_aprobado = pd.crosstab(df["higher"], df["aprobado"], normalize="index").round(3)
higher_aprobado

In [ ]:
# Barras con la tasa de aprobación de cada grupo; rot=0 deja las etiquetas horizontales
higher_aprobado["sí"].plot(kind="bar", rot=0, label="tasa de aprobación")

# Línea horizontal con la tasa de aprobación de todos los estudiantes (paso 8)
plt.axhline(0.671, color="black", linestyle="--", label="todos los estudiantes (67.1 %)")
plt.title("Tasa de aprobación según si quiere estudios superiores")
plt.xlabel("Quiere estudios superiores (higher)")
plt.ylabel("Tasa de aprobación")
plt.legend()
plt.show()

* **`G2` y `G3` (numérica y numérica):** los puntos forman casi una línea recta: quien tuvo buena nota en el
  segundo periodo tiene buena nota final. La excepción es una fila de puntos en `G3 = 0`, con `G2` entre 0 y 10:
  son los estudiantes sin nota final del paso 4.
* **`failures` y `G3` (categórica y numérica):** la mediana de la nota final baja con cada materia perdida: **11
  con 0 materias** (312 estudiantes), 9 con 1 (50), 8 con 2 (17) y 7 con 3 (16). Los grupos con materias perdidas
  son pequeños, pero el patrón es claro.
* **`higher` y `aprobado` (categórica y categórica):** de los estudiantes que quieren estudios superiores aprueba
  **68.8 %**; de los que no quieren, **35.0 %**, muy por debajo del 67.1 % general. El grupo `no` tiene solo 20
  estudiantes, así que la cifra puede cambiar mucho con pocos casos.
* Un patrón no es una causa. Haber perdido materias puede ser señal de dificultades que también explican la nota
  baja; el EDA muestra la relación, pero no dice qué columna la produce.

## Paso 10. Correlaciones

In [ ]:
# Matriz de correlaciones de Pearson entre todas las columnas numéricas
corr = df.select_dtypes("number").corr()

# Figura grande para que se lean los 16 x 16 valores
plt.figure(figsize=(12, 10))

# annot=True escribe el valor en cada celda; vmin y vmax fijan la escala de colores de -1 a 1
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlaciones entre columnas numéricas")
plt.show()

In [ ]:
corr["G3"].drop("G3").sort_values().round(2)  # se quita la correlación de G3 consigo misma (1.00)

In [ ]:
# Misma correlación, primero con todos los estudiantes y después sin los de G3 = 0 (con_nota, paso 8)
print("con todos:   ", round(df["absences"].corr(df["G3"]), 2))
print("sin G3 = 0:  ", round(con_nota["absences"].corr(con_nota["G3"]), 2))

### Gráfico interactivo (plotly)

In [ ]:
# Un punto por estudiante, con color según aprobó o no;
# hover_data define qué columnas aparecen al pasar el cursor
fig = px.scatter(
    df, x="absences", y="G3", color="aprobado",
    hover_data=["school", "age", "G2"],
    labels={"absences": "Ausencias", "G3": "Nota final (G3)", "aprobado": "Aprobado"},
    title="Ausencias y nota final",
)
fig.show()

### ✍️ 10. Respuesta

* **Pares más fuertes:** `G2` y `G3` (**0.90**), `G1` y `G2` (0.85) y `G1` y `G3` (0.80). Las tres notas del año
  llevan casi la misma información: son medidas del mismo estudiante en la misma materia.
* **Otros pares fuertes:** `Dalc` y `Walc` (0.65): quien bebe entre semana también bebe el fin de semana. `Medu` y
  `Fedu` (0.62): los padres suelen tener un nivel educativo parecido.
* **Pares más débiles:** varios están en 0.00, por ejemplo `Fedu` y `famrel`: la educación del padre no se mueve
  con la calidad de la relación familiar.
* **Fila de la variable objetivo:** sin contar `G1` y `G2`, la correlación más fuerte con `G3` es la de `failures`
  (**−0.36**), seguida de `Medu` (0.22), `age` (−0.16) y `goout` (−0.13).
* **Una relación escondida:** con todos los estudiantes, `absences` y `G3` tienen correlación **0.03**, como si
  no se relacionaran. Sin los 38 ceros es **−0.21**: más ausencias, nota un poco más baja. Los ceros sin nota
  final, todos con 0 ausencias, esconden esa relación.

## Paso 11. Hallazgos
| # | Hallazgo | Evidencia | Paso |
|---|---|---|---|
| 1 | Los ceros de `G3` parecen "sin nota", no una nota de 0 | 38 estudiantes (9.6 %); los 38 tienen 0 ausencias y 13 tienen también `G2 = 0` | 4, 8 |
| 2 | No hay faltantes ni duplicados visibles, pero tampoco un código de estudiante | 0 faltantes, 0 filas duplicadas | 4 |
| 3 | 11 columnas numéricas son escalas, y `failures` no coincide con su documentación | la documentación dice que el máximo es 4; en los datos es 3 | 3 |
| 4 | Las ausencias tienen sesgo a la derecha | media 5.71, mediana 4; 15 atípicos entre 21 y 75 | 5, 6, 7 |
| 5 | Las categorías están bien escritas | 17 columnas de texto, ninguna cambia con `strip` y `lower` | 6 |
| 6 | 2 de cada 3 estudiantes aprueban | 67.1 % con `G3` ≥ 10 | 8 |
| 7 | A más materias perdidas antes, menor nota final | medianas de 11, 9, 8 y 7 para 0, 1, 2 y 3 materias | 9 |
| 8 | Quien no quiere estudios superiores aprueba mucho menos | 35.0 % contra 68.8 % (grupo `no` de 20 estudiantes) | 9 |
| 9 | Las tres notas del año dicen casi lo mismo | correlación de 0.90 entre `G2` y `G3`, y de 0.80 entre `G1` y `G3` | 9, 10 |
| 10 | Los ceros esconden la relación entre ausencias y nota | correlación de 0.03 con todos y de −0.21 sin `G3 = 0` | 10 |

## Paso 12. Próximos pasos

**Limpieza pendiente**

* **`G3 = 0` y `G2 = 0`:** confirmar con la documentación o con el colegio si significan "no presentó". Si es así,
  marcarlos como faltantes o analizarlos como un grupo aparte, porque bajan la media y esconden relaciones
  (hallazgos 1 y 10).
* **Columnas sí/no:** convertir las 8 columnas `yes`/`no` a verdadero/falso, para poder sumarlas y promediarlas.
* **`failures`:** documentar que el 3 puede significar "3 o más", porque no coincide con la documentación.

**Preguntas para después**

* ¿Las materias perdidas antes (`failures`) explican la nota final, o son una señal de otras dificultades que
  también la explican?
* ¿Se puede estimar la nota final sin `G1` y `G2`? Los autores del dataset advierten que es más difícil, pero más
  útil, porque permitiría actuar antes de que termine el año.
* ¿Por qué los 38 estudiantes sin nota final tienen todos 0 ausencias? ¿Se retiraron al comienzo del año?
* ¿Los patrones se repiten en la asignatura de portugués (`student-por.csv`)?